In [15]:
! pip install git+https://github.com/openai/whisper.git -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [16]:
# frontend Application for building web app
! pip install gradio -q

In [17]:
import gradio as gr
import os
import sys
import subprocess
#from moviepy.editor import VideoFileClip


In [18]:
import whisper

model = whisper.load_model("medium")

In [24]:
# conversion of the video into mp3 from any language to English
def video2mp3(video_file, output_ext="mp3"):
    filename, ext = os.path.splitext(video_file)
    subprocess.call(["ffmpeg", "-y", "-i", video_file, f"{filename}.{output_ext}"],
                    stdout=subprocess.DEVNULL,
                    stderr=subprocess.STDOUT)
    return f"{filename}.{output_ext}"


In [25]:
#input_video = 'this+invention+makes+getting+out+of+bed+so+much+easier..mp4'

input_video = '/content/me.26.24.mp4'

In [26]:
audio_file = video2mp3(input_video)

In [27]:
from IPython.display import Audio
Audio(audio_file)

In [28]:
# translation
def translate(audio):

    options = dict(beam_size=5, best_of=5)
    translate_options = dict(task="translate", **options)
    result = model.transcribe(audio_file,**translate_options)
    return result




In [29]:
result = translate(audio_file)

In [30]:
print(result["text"])

 Domain Expansion Millwitch Shrine Okay


In [31]:
from IPython.display import HTML
from base64 import b64encode
mp4 = open(input_video,'rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
HTML("""
<video width=400 controls>
      <source src="%s" type="video/mp4">
</video>
""" % data_url)

In [32]:
output_dir = '/content/'
audio_path = audio_file.split(".")[0]

In [33]:
def write_srt(segments, filename):
    def format_time(seconds):
        hrs = int(seconds // 3600)
        mins = int((seconds % 3600) // 60)
        secs = seconds % 60
        return f"{hrs:02}:{mins:02}:{secs:06.3f}".replace('.', ',')

    with open(filename, "w") as f:
        for i, seg in enumerate(segments):
            f.write(f"{i+1}\n")
            f.write(f"{format_time(seg['start'])} --> {format_time(seg['end'])}\n")
            f.write(f"{seg['text']}\n\n")

In [ ]:
# with open(os.path.join(output_dir, audio_path + ".srt"), "w") as vtt:
#     write_vtt(result["segments"], file=vtt)


In [ ]:
# subtitle = audio_path + ".vtt"
# output_video = audio_path + "_subtitled.mp4"

In [34]:
subtitle_path = os.path.join(output_dir, audio_path + ".srt")

write_srt(result["segments"], subtitle_path)

In [35]:
output_video = audio_path + "_subtitled.mp4"

In [36]:
os.system(f"ffmpeg -i {input_video} -vf subtitles={subtitle_path} {output_video}")


# subprocess.call(["ffmpeg", "-i", input_video , "-vf", f"subtitles={subtitle}", f"{output_video}"],
#                 stdout=subprocess.DEVNULL,
#                 stderr=subprocess.STDOUT)

0

In [37]:
from IPython.display import HTML
from base64 import b64encode
mp4 = open(output_video,'rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
HTML("""
<video width=400 controls>
      <source src="%s" type="video/mp4">
</video>
""" % data_url)

In [39]:
!apt install ffmpeg

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 42 not upgraded.


In [43]:

import gradio as gr
import os
import sys
import subprocess
#from moviepy.editor import VideoFileClip

import whisper


model = whisper.load_model("large")

def video2mp3(video_file, output_ext="mp3"):
    filename, ext = os.path.splitext(video_file)
    subprocess.call(["ffmpeg", "-y", "-i", video_file, f"{filename}.{output_ext}"],
                    stdout=subprocess.DEVNULL,
                    stderr=subprocess.STDOUT)
    return f"{filename}.{output_ext}"

def write_srt(segments, filename):
    def format_time(seconds):
        hrs = int(seconds // 3600)
        mins = int((seconds % 3600) // 60)
        secs = seconds % 60
        return f"{hrs:02}:{mins:02}:{secs:06.3f}".replace('.', ',')

    with open(filename, "w") as f:
        for i, seg in enumerate(segments):
            f.write(f"{i+1}\n")
            f.write(f"{format_time(seg['start'])} --> {format_time(seg['end'])}\n")
            f.write(f"{seg['text']}\n\n")
def translate(input_video):


    if isinstance(input_video, dict):
        input_video = input_video["name"]

    audio_file = video2mp3(input_video)

    options = dict(beam_size=5, best_of=5)
    translate_options = dict(task="translate", **options)

    result = model.transcribe(audio_file, **translate_options)

    output_dir = '/content/'
    audio_path = os.path.splitext(os.path.basename(audio_file))[0]

    subtitle_path = os.path.join(output_dir, audio_path + ".srt")
    write_srt(result["segments"], subtitle_path)

    output_video = os.path.join(output_dir, audio_path + "_subtitled.mp4")


    os.system(f'ffmpeg -y -i "{input_video}" -vf subtitles="{subtitle_path}" "{output_video}"')

    return output_video



title = "Add Text/Caption to your YouTube Shorts - MultiLingual"

block = gr.Blocks()

with block:

    with gr.Group():
      with gr.Column():



            with gr.Row():

                inp_video = gr.Video(label="Input Video")
                op_video = gr.Video()
            btn = gr.Button("Generate Subtitle Video")






      btn.click(translate, inputs=[inp_video], outputs=[op_video])

      gr.HTML('''
        <div class="footer">
                    <p>Model by <a href="https://github.com/openai/whisper" style="text-decoration: underline;" target="_blank">OpenAI</a> - Gradio App by <a href="https://www.linkedin.com/in/charanjot-kaur-arora/" style="text-decoration: underline;" target="_blank">Charanjot Kaur</a>
                    </p>
        </div>
        ''')

block.launch(debug = True)


OutOfMemoryError: CUDA out of memory. Tried to allocate 102.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 19.81 MiB is free. Including non-PyTorch memory, this process has 14.54 GiB memory in use. Of the allocated memory 14.07 GiB is allocated by PyTorch, and 340.83 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)